# Zalo AI Challenge 2026 — LifeMate: Complete Baseline Notebook

Notebook này chứa **toàn bộ mã nguồn hoàn chỉnh (standalone/self-contained)** của mô hình Baseline cho cuộc thi **ZAC 2026 - LifeMate**, không phụ thuộc vào bất kỳ file script ngoài nào.

### 1. Kiến trúc Tổng thể (End-to-End Architecture):
- **Đầu vào (Input)**: File âm thanh giọng nói (`.wav` 16kHz mono) + Ngữ cảnh `context` (`current_time`, `user_profile`, `installed_apps`) + Lịch sử hội thoại `history`.
- **Module ASR**: Nhận diện giọng nói tiếng Việt thành văn bản (`audio_text`).
- **Module DateTimeResolver**: Chuyển đổi thời gian tự nhiên tiếng Việt (`sáng mai`, `tối nay`, `5 rưỡi`, ...) thành chuẩn ISO-8601.
- **Agentic Decision Engine**: Ánh xạ câu lệnh sang các thao tác Tool (`smart_home`, `alarm`, `reminders`, `weather`, `contacts`, `sys.ask_missing_info`).
- **Runtime Môi trường (`zaic_tools.ToolEnv`)**: Thực thi tuần tự các tool call trên `state.json` và cập nhật `state_hash`.
- **Đầu ra (Output)**: File nộp bài `submission.jsonl` chuẩn định dạng BTC.

import io
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import urllib.request
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

# 1. Kiểm tra thiết bị phần cứng & PyTorch (Safe check chống lỗi triton conflict trên Python 3.14)
device = "cpu"
try:
    import torch
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    print(f"PyTorch Version: {torch.__version__} | Device: {device}")
except Exception as e:
    print(f"[Lưu ý] PyTorch gặp xung đột hoặc chưa sẵn sàng ({type(e).__name__}).")
    print("Mô hình Baseline LifeMate thuần logic không cần PyTorch và vẫn chạy 100% trơn tru trên CPU.")

# 2. Cấu hình Link GitHub Repo Dataset
PROJECT_ROOT = Path(".").resolve()
GITHUB_REPO_URL = os.getenv("ZALOAI_REPO_URL", "https://github.com/vdhwy/ZALOAI.git")
GITHUB_RAW_BASE = os.getenv(
    "ZALOAI_RAW_BASE", "https://raw.githubusercontent.com/vdhwy/ZALOAI/main/train+public_test"
)
DATA_DIR = PROJECT_ROOT / "train+public_test"

def ensure_dataset(
    repo_url: str = GITHUB_REPO_URL,
    data_dir: Path = DATA_DIR,
    branch: str = "main",
) -> Path:
    """Đảm bảo dataset tồn tại ở local; nếu chưa có, tự động shallow clone (--depth 1) từ GitHub."""
    data_path = Path(data_dir).resolve()
    train_marker = data_path / "train" / "train.jsonl"
    test_marker = data_path / "public_test" / "public_test.jsonl"

    if train_marker.is_file() and test_marker.is_file():
        print(f"[Dataset] Dữ liệu đã sẵn sàng tại: {data_path}")
        return data_path

    print(f"[Dataset] Chưa tìm thấy dữ liệu local. Đang tải từ GitHub repo: {repo_url} (branch: {branch})...")
    temp_dir = data_path.parent / "_temp_repo_clone"
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)

    try:
        cmd = ["git", "clone", "--depth", "1", "-b", branch, repo_url, str(temp_dir)]
        subprocess.run(cmd, check=True)
        candidate = temp_dir / "train+public_test"
        if candidate.is_dir():
            if data_path.exists():
                shutil.rmtree(data_path, ignore_errors=True)
            shutil.move(str(candidate), str(data_path))
        else:
            if data_path.exists():
                shutil.rmtree(data_path, ignore_errors=True)
            shutil.move(str(temp_dir), str(data_path))
        print(f"[Dataset] Tải và giải nén thành công tại: {data_path}")
    except Exception as e:
        print(f"[Dataset] Lỗi khi clone ({e}). Có thể đọc trực tiếp bằng stream GitHub Raw.")
    finally:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)
    return data_path

# 3. Kích hoạt dataset & nạp zaic_tools vào sys.path
data_root = ensure_dataset(repo_url=GITHUB_REPO_URL, data_dir=DATA_DIR)
for p in [data_root / "train", data_root / "public_test"]:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from zaic_tools import ToolEnv, schema_validate
from zaic_tools.utils import state_hash
print("Nạp zaic_tools thành công!")


In [ ]:
import io
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import urllib.request
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

# 1. Kiểm tra thiết bị phần cứng & PyTorch (Safe check chống lỗi triton conflict trên Python 3.14)
device = 'cpu'
try:
    import torch
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
    elif torch.cuda.is_available():
        device = torch.device('cuda')
    print(f"PyTorch Version: {torch.__version__} | Device: {device}")
except Exception as e:
    print(f"[Lưu ý] PyTorch gặp xung đột hoặc chưa sẵn sàng ({type(e).__name__}).")
    print("Mô hình Baseline LifeMate thuần logic không cần PyTorch và vẫn chạy 100% trơn tru trên CPU.")

# 2. Cấu hình Link GitHub Repo Dataset
PROJECT_ROOT = Path(".").resolve()
GITHUB_REPO_URL = os.getenv("ZALOAI_REPO_URL", "https://github.com/vdhwy/ZALOAI.git")
GITHUB_RAW_BASE = os.getenv(
    "ZALOAI_RAW_BASE", "https://raw.githubusercontent.com/vdhwy/ZALOAI/main/train+public_test"
)
DATA_DIR = PROJECT_ROOT / "train+public_test"


def ensure_dataset(
    repo_url: str = GITHUB_REPO_URL,
    data_dir: Path = DATA_DIR,
    branch: str = "main",
) -> Path:
    """Đảm bảo dataset tồn tại ở local; nếu chưa có, tự động shallow clone (--depth 1) từ GitHub."""
    data_path = Path(data_dir).resolve()
    train_marker = data_path / "train" / "train.jsonl"
    test_marker = data_path / "public_test" / "public_test.jsonl"

    if train_marker.is_file() and test_marker.is_file():
        print(f"[Dataset] Dữ liệu đã sẵn sàng tại: {data_path}")
        return data_path

    print(f"[Dataset] Chưa tìm thấy dữ liệu local. Đang tải từ GitHub repo: {repo_url} (branch: {branch})...")
    temp_dir = data_path.parent / "_temp_repo_clone"
    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)

    try:
        cmd = ["git", "clone", "--depth", "1", "-b", branch, repo_url, str(temp_dir)]
        subprocess.run(cmd, check=True)
        candidate = temp_dir / "train+public_test"
        if candidate.is_dir():
            if data_path.exists():
                shutil.rmtree(data_path, ignore_errors=True)
            shutil.move(str(candidate), str(data_path))
        else:
            if data_path.exists():
                shutil.rmtree(data_path, ignore_errors=True)
            shutil.move(str(temp_dir), str(data_path))
        print(f"[Dataset] Tải và giải nén thành công tại: {data_path}")
    except Exception as e:
        print(f"[Dataset] Lỗi khi clone ({e}). Có thể đọc trực tiếp bằng stream GitHub Raw.")
    finally:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)
    return data_path


# 3. Kích hoạt dataset & nạp zaic_tools vào sys.path
data_root = ensure_dataset(repo_url=GITHUB_REPO_URL, data_dir=DATA_DIR)
for p in [data_root / "train", data_root / "public_test"]:
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from zaic_tools import ToolEnv, schema_validate
from zaic_tools.utils import state_hash
print("Nạp zaic_tools thành công!")


## Bước 2: Trình Đọc Dữ Liệu Trong Suốt (Local & GitHub Raw Stream)

In [ ]:
def open_dataset(path_or_url: str | Path) -> List[str]:
    """Đọc các dòng JSONL trong suốt từ file nội bộ hoặc từ đường link HTTP/GitHub Raw URL."""
    s = str(path_or_url)
    if s.startswith("http://") or s.startswith("https://"):
        req = urllib.request.Request(s, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            return resp.read().decode("utf-8").splitlines()
    with open(path_or_url, "r", encoding="utf-8") as f:
        return f.read().splitlines()


# Kiểm tra đọc thử 1 dòng đầu tiên từ file local
lines = open_dataset(data_root / "train" / "train.jsonl")
sample = json.loads(lines[0])
print(f"Sample ID: {sample['id']}")
print(f"Audio Path: {sample['audio']}")
print(f"Audio Text: {sample['audio_text']}")
print(f"Context Time: {sample['context']['current_time']}")
print(f"Installed Apps: {list(sample['context']['installed_apps'].keys())}")
print("\nGround Truth Tool Calls:")
for c in sample['label']['tool_calls']:
    print(f"  -> {c['name']}({c['arguments']})")

## Bước 3: Thử Nghiệm Môi Trường ToolEnv & Tính State Hash

In [ ]:
# Khởi tạo phiên ToolEnv cho mẫu train_0001
env = ToolEnv("train", sample["id"], root=data_root)
init_hash = env.get_state_hash()
print(f"Initial state_hash: {init_hash}")

# Chạy tuần tự các tool call của nhãn
for call in sample["label"]["tool_calls"]:
    res = env.call(call["name"], call["arguments"])
    print(f"Executed: {call['name']} -> ok: {res.get('ok')}")

final_hash = env.get_state_hash()
print(f"Final state_hash:   {final_hash}")
print(f"Khớp với nhãn: {final_hash == state_hash(env.get_state())}")

## Bước 4: Các Module Xử Lý Tiền Đề (ASR & DateTimeResolver)

In [ ]:
class SimpleASR:
    """Module nhận diện giọng nói tiếng Việt, hỗ trợ fallback text khi có sẵn."""

    def __init__(self, model_name: Optional[str] = None, device: str = "cpu"):
        self.model_name = model_name
        self.device = device
        self._pipe = None

    def load_model(self):
        if self._pipe is None and self.model_name:
            try:
                from transformers import pipeline
                print(f"Loading ASR pipeline {self.model_name} on {self.device}...")
                self._pipe = pipeline("automatic-speech-recognition", model=self.model_name, device=self.device)
            except Exception as e:
                print(f"Warning: Không thể nạp ASR ({e}). Fallback sang text.")

    def transcribe(self, audio_path: Path, fallback_text: str = "") -> str:
        if fallback_text:
            return fallback_text
        if self._pipe is not None and audio_path.is_file():
            try:
                res = self._pipe(str(audio_path))
                return res.get("text", "").strip()
            except Exception as e:
                print(f"ASR error ({audio_path}): {e}")
        return ""


class DateTimeResolver:
    """Parser thời gian tiếng Việt tự nhiên sang chuẩn ISO-8601 dựa trên current_time."""

    @staticmethod
    def parse_current_time(current_time_str: str) -> datetime:
        return datetime.fromisoformat(current_time_str[:19])

    @classmethod
    def resolve_datetime(cls, text: str, current_time_str: str, tz_offset: str = "+07:00") -> Optional[str]:
        now = cls.parse_current_time(current_time_str)
        target_date = now.date()
        text_lower = text.lower()

        # Xử lý ngày lệch
        if any(w in text_lower for w in ["ngày mai", "sáng mai", "tối mai", "chiều mai", "mai"]):
            target_date += timedelta(days=1)
        elif "ngày kia" in text_lower:
            target_date += timedelta(days=2)

        # Ngày cụ thể (vd: ngày 19)
        day_match = re.search(r"ngày\s+(\d{1,2})", text_lower)
        if day_match:
            try:
                d = int(day_match.group(1))
                target_date = target_date.replace(day=d)
            except Exception:
                pass

        # Xử lý giờ, phút
        hour = 7
        minute = 0

        rưỡi_match = re.search(r"(\d{1,2})\s*(?:giờ|h)?\s*rưỡi", text_lower)
        colon_match = re.search(r"(\d{1,2})[:h](\d{2})", text_lower)
        hour_match = re.search(r"(\d{1,2})\s*(?:giờ|h)", text_lower)

        if rưỡi_match:
            hour, minute = int(rưỡi_match.group(1)), 30
        elif colon_match:
            hour, minute = int(colon_match.group(1)), int(colon_match.group(2))
        elif hour_match:
            hour, minute = int(hour_match.group(1)), 0
        elif "tám rưỡi" in text_lower:
            hour, minute = 8, 30
        elif "tám giờ" in text_lower:
            hour, minute = 8, 0
        elif "sáu rưỡi" in text_lower:
            hour, minute = 6, 30
        elif "sáu giờ" in text_lower:
            hour, minute = 6, 0
        elif "năm rưỡi" in text_lower:
            hour, minute = 5, 30
        elif "năm giờ" in text_lower:
            hour, minute = 5, 0
        elif "bảy giờ" in text_lower:
            hour, minute = 7, 0

        if ("tối" in text_lower or "chiều" in text_lower) and hour < 12:
            hour += 12

        dt = datetime(target_date.year, target_date.month, target_date.day, hour, minute, 0)
        return dt.isoformat() + tz_offset

print("DateTimeResolver khởi tạo thành công!")
test_time = DateTimeResolver.resolve_datetime("đặt báo thức 5 rưỡi sáng mai nhé", "2026-11-10T08:00:00+07:00")
print("Test parse '5 rưỡi sáng mai' ->", test_time)

## Bước 5: Triển Khai Mô Hình Baseline Agent

Agent thực hiện suy luận, đọc danh mục `installed_apps` và tạo chuỗi `tool_calls` an toàn đã qua kiểm duyệt schema.

In [ ]:
class BaseAgent:
    def predict(self, item: Dict[str, Any], env: Any) -> List[Dict[str, Any]]:
        raise NotImplementedError


class HeuristicBaselineAgent(BaseAgent):
    """Baseline Agent thông minh dựa trên khớp mẫu từ khóa ngữ nghĩa và đối chiếu catalog."""

    def __init__(self):
        self.time_resolver = DateTimeResolver()

    def predict(self, item: Dict[str, Any], env: Any) -> List[Dict[str, Any]]:
        text = item.get("audio_text", "").lower()
        context = item.get("context", {})
        now_str = context.get("current_time", "2026-11-10T08:00:00+07:00")
        allowed = env.allowed
        states = env.states

        calls = []

        # 1. Thời tiết (Weather)
        if "weather.get_forecast" in allowed:
            if any(w in text for w in ["thời tiết", "mưa", "nhiệt độ", "dự báo"]):
                target_date = self.time_resolver.parse_current_time(now_str).date()
                if "mai" in text:
                    target_date += timedelta(days=1)
                date_str = target_date.isoformat()
                env.call("weather.get_forecast", {"date": date_str})
                calls.append({"name": "weather.get_forecast", "arguments": {"date": date_str}})

        # 2. Nhà thông minh (Smart Home)
        if "smart_home.light_set" in allowed:
            sh_state = states.get("smart_home", {})
            lights = [d for d in sh_state.get("devices", []) if d.get("type") == "light"]
            if lights:
                target_device = lights[0]["id"]
                for d in lights:
                    name, room = d.get("name", "").lower(), d.get("room", "").lower()
                    if (name and name in text) or (room and room in text):
                        target_device = d["id"]
                        break
                if any(w in text for w in ["bật đèn", "mở đèn", "bật ánh sáng"]):
                    env.call("smart_home.light_set", {"device_id": target_device, "power": "on"})
                    calls.append({"name": "smart_home.light_set", "arguments": {"device_id": target_device, "power": "on"}})
                elif any(w in text for w in ["tắt đèn", "đóng đèn"]):
                    env.call("smart_home.light_set", {"device_id": target_device, "power": "off"})
                    calls.append({"name": "smart_home.light_set", "arguments": {"device_id": target_device, "power": "off"}})

        if "smart_home.fan_set" in allowed:
            sh_state = states.get("smart_home", {})
            fans = [d for d in sh_state.get("devices", []) if d.get("type") == "fan"]
            if fans:
                dev_id = fans[0]["id"]
                if any(w in text for w in ["bật quạt", "mở quạt"]):
                    env.call("smart_home.fan_set", {"device_id": dev_id, "power": "on"})
                    calls.append({"name": "smart_home.fan_set", "arguments": {"device_id": dev_id, "power": "on"}})
                elif any(w in text for w in ["tắt quạt"]):
                    env.call("smart_home.fan_set", {"device_id": dev_id, "power": "off"})
                    calls.append({"name": "smart_home.fan_set", "arguments": {"device_id": dev_id, "power": "off"}})

        if "smart_home.get_status" in allowed and not calls:
            if any(w in text for w in ["trạng thái nhà", "kiểm tra nhà", "các thiết bị"]):
                env.call("smart_home.get_status", {})
                calls.append({"name": "smart_home.get_status", "arguments": {}})

        # 3. Báo thức (Alarm)
        if "alarm.create" in allowed:
            if any(w in text for w in ["báo thức", "đặt báo thức", "hẹn giờ"]):
                resolved_time = self.time_resolver.resolve_datetime(text, now_str)
                if resolved_time:
                    args = {"time": resolved_time}
                    if env.validate("alarm.create", args).get("ok"):
                        env.call("alarm.create", args)
                        calls.append({"name": "alarm.create", "arguments": args})

        if "alarm.list" in allowed and "danh sách báo thức" in text:
            env.call("alarm.list", {})
            calls.append({"name": "alarm.list", "arguments": {}})

        # 4. Lịch nhắc (Reminders)
        if "reminders.create" in allowed:
            if any(w in text for w in ["nhắc tôi", "nhắc nhở", "nhớ nhắc"]):
                resolved_time = self.time_resolver.resolve_datetime(text, now_str)
                content = "lịch nhắc"
                m = re.search(r"nhắc\s+(?:tôi|mình|dùm chị|giúp)?\s*(.*?)(?:\s+lúc|\s+vào|\s+nhé|\.|$)", text)
                if m and m.group(1).strip():
                    content = m.group(1).strip()
                args = {"content": content, "time": resolved_time or now_str}
                if env.validate("reminders.create", args).get("ok"):
                    env.call("reminders.create", args)
                    calls.append({"name": "reminders.create", "arguments": args})

        # 5. Danh bạ (Contacts Search)
        if "contacts.search" in allowed:
            contact_match = re.search(r"(?:nhắn|gọi|hỏi|báo cho)\s+(?:sếp|anh|chị|bạn|bố|mẹ)?\s*([A-ZÀ-Ỹa-zà-ỹ0-9\s]+)", text)
            if contact_match and not any(c["name"] == "contacts.search" for c in calls):
                query = contact_match.group(1).strip().split()[0] if contact_match.group(1) else ""
                if query and query not in ["tôi", "mình", "ai"]:
                    env.call("contacts.search", {"query": query})
                    calls.append({"name": "contacts.search", "arguments": {"query": query}})

        # 6. Thiếu thông tin bắt buộc (sys.ask_missing_info)
        if "phone.call" in allowed and "sys.ask_missing_info" in allowed:
            if any(w in text for w in ["gọi cho", "gọi điện"]) and not any(c["name"] == "phone.call" for c in calls):
                args = {"tool_name": "phone.call", "params": ["contact_id"]}
                env.call("sys.ask_missing_info", args)
                calls.append({"name": "sys.ask_missing_info", "arguments": args})

        return calls


print("Khởi tạo HeuristicBaselineAgent thành công!")

## Bước 6: Đánh Giá Baseline trên Tập Train (Đo State Hash Accuracy)

In [ ]:
def evaluate_agent(
    agent: BaseAgent,
    data_source: str | Path,
    data_root: Path = DATA_DIR,
    split: str = "train",
    max_samples: Optional[int] = None,
) -> Dict[str, float]:
    """Đánh giá agent trên tập train: so sánh state_hash dự đoán với state_hash nhãn thực tế."""
    print(f"\n--- Đang đánh giá Agent trên {split} (source: {data_source}) ---")
    correct_hash = 0
    total = 0
    total_calls_pred = 0
    total_calls_target = 0
    exact_tool_names = 0

    lines = open_dataset(data_source)
    for idx, line in enumerate(lines):
        if not line.strip():
            continue
        if max_samples and idx >= max_samples:
            break

        item = json.loads(line)
        sample_id = item["id"]

        # 1. Oracle environment: chạy nhãn để lấy hash chuẩn
        oracle_env = ToolEnv(split, sample_id, root=data_root)
        for call in item["label"]["tool_calls"]:
            oracle_env.call(call["name"], call.get("arguments", {}))
        target_hash = oracle_env.get_state_hash()
        target_tools = [c["name"] for c in item["label"]["tool_calls"]]

        # 2. Prediction environment: agent dự đoán
        pred_env = ToolEnv(split, sample_id, root=data_root)
        agent.predict(item, pred_env)
        pred_hash = pred_env.get_state_hash()
        pred_calls = pred_env.get_tool_calls()
        pred_tools = [c["name"] for c in pred_calls]

        if pred_hash == target_hash:
            correct_hash += 1
        if pred_tools == target_tools:
            exact_tool_names += 1

        total_calls_pred += len(pred_calls)
        total_calls_target += len(target_tools)
        total += 1

        if total % 50 == 0:
            print(f"Đã đánh giá {total} mẫu | Accuracy: {correct_hash/total*100:.2f}%")

    acc = correct_hash / total if total > 0 else 0.0
    tool_acc = exact_tool_names / total if total > 0 else 0.0

    print("\n--- KẾT QUẢ ĐÁNH GIÁ BASELINE ---")
    print(f"Số lượng mẫu đánh giá: {total}")
    print(f"State Hash Accuracy:   {acc*100:.2f}% ({correct_hash}/{total})")
    print(f"Exact Tool Match:       {tool_acc*100:.2f}% ({exact_tool_names}/{total})")
    print(f"Tổng số tool call (Dự đoán / Nhãn): {total_calls_pred} / {total_calls_target}")
    return {"accuracy": acc, "tool_accuracy": tool_acc, "total": total}


# Chạy đánh giá trên 100 mẫu đầu tiên
agent = HeuristicBaselineAgent()
train_source = data_root / "train" / "train.jsonl"
results = evaluate_agent(agent, data_source=train_source, data_root=data_root, max_samples=100)

## Bước 7: Dự Đoán Trên Public Test & Tạo File Nộp Bài (`submission.jsonl`)

In [ ]:
def generate_submission(
    agent: BaseAgent,
    test_source: str | Path,
    output_path: Path,
    data_root: Path = DATA_DIR,
    split: str = "public_test",
) -> None:
    """Chạy dự đoán trên split test và xuất ra file submission.jsonl đúng chuẩn kiểm duyệt."""
    print(f"\n--- Đang sinh file submission cho {split} -> {output_path} ---")
    count = 0
    all_calls = []

    lines = open_dataset(test_source)
    with open(output_path, "w", encoding="utf-8") as fout:
        for line in lines:
            if not line.strip():
                continue
            item = json.loads(line)
            sample_id = item["id"]

            env = ToolEnv(split, sample_id, root=data_root)
            agent.predict(item, env)

            calls = env.get_tool_calls()
            all_calls.extend(calls)
            record = {
                "id": sample_id,
                "tool_calls": calls,
                "state_hash": env.get_state_hash(),
            }
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1

    print(f"Đã ghi thành công {count} dòng vào file: {output_path}.")

# Kiểm tra schema validation với danh mục tools của split
    catalog_file = data_root / split / f"{split}_tools_catalog.json"
    if catalog_file.is_file():
        print("Đang kiểm duyệt Schema Validation cho bài nộp...")
        catalog = json.load(open(catalog_file, encoding="utf-8"))
        val_res = schema_validate(all_calls, catalog)
        errors = [v for v in val_res if not v.get("ok")]
        print(f"Kết quả: {len(all_calls)} tool calls được kiểm tra, {len(errors)} lỗi schema.")
        if errors:
            print("Các lỗi đầu tiên:", errors[:3])
        else:
            print("🎉 Bài nộp đạt chuẩn 100% Schema, sẵn sàng nộp lên Leaderboard!")


# Chạy sinh submission cho 634 mẫu public test
test_source = data_root / "public_test" / "public_test.jsonl"
output_file = PROJECT_ROOT / "submission.jsonl"
generate_submission(agent, test_source=test_source, output_path=output_file, data_root=data_root)

# Hiển thị 3 dòng mẫu đầu tiên của file submission
print("\n--- 3 Dòng Mẫu Đầu Tiên Của submission.jsonl ---")
with open(output_file, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip())